In [1]:
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt

engine = create_engine("mysql+mysqlconnector://root:@localhost:3306/dsm_final_project")

engine.dispose()

In [4]:
from pathlib import Path

engine = create_engine("mysql+mysqlconnector://root:@localhost:3306/dsm_final_project")

csv_files = [
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\demographics_st.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\dropout_st.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\employment_st.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\enrolment_st.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\ger_st.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\high_st_share.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\household_type_rural.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\literacy.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\low_literacy_districts.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\mgnreg_st.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\poverty_st.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\sc_st_residence.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\tribal_villages.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\tribe_socioeconomic.csv",
]

for file_path in csv_files:
    table_name = Path(file_path).stem
    df = pd.read_csv(file_path)
    df.to_sql(table_name, con=engine, if_exists="replace", index=False)

engine.dispose()

In [3]:
df = pd.read_sql("SELECT * FROM demographics_st", engine)
print(df.head())

                         state  year additional_information  total_population  \
0  Andaman and Nicobar Islands  2011                   None          380581.0   
1               Andhra Pradesh  2011                   None        84580777.0   
2            Arunachal Pradesh  2011                   None         1383727.0   
3                        Assam  2011                   None        31205576.0   
4                        Bihar  2011                   None       104099452.0   

   state_share_india_population_pct  st_population  \
0                              6.86        28530.0   
1                             10.98      5918073.0   
2                             26.03       951821.0   
3                             17.07      3884371.0   
4                             25.42      1336573.0   

   state_share_india_st_population_pct  st_share_state_population_pct  \
0                                 3.19                           7.50   
1                                17.79    

In [9]:
schema_df = pd.read_sql(
    """
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable,
        column_key,
        column_default,
        extra
    FROM information_schema.columns
    WHERE table_schema = DATABASE()
    ORDER BY table_name, ordinal_position
    """,
    engine,
)

for table_name, group in schema_df.groupby("table_name"):
    print(f"\nTable: {table_name}")
    print(group[["column_name", "data_type", "is_nullable", "column_key", "column_default", "extra"]].to_string(index=False))


Table: demographics_st
                        column_name data_type is_nullable column_key column_default extra
                              state      text         YES                      NULL      
                               year    bigint         YES                      NULL      
             additional_information      text         YES                      NULL      
                   total_population    double         YES                      NULL      
   state_share_india_population_pct    double         YES                      NULL      
                      st_population    double         YES                      NULL      
state_share_india_st_population_pct    double         YES                      NULL      
      st_share_state_population_pct    double         YES                      NULL      
                 decadal_growth_pct    double         YES                      NULL      

Table: dropout_st
              column_name data_type is_nullable column_ke

In [23]:
query = "SELECT * FROM demographics_st"
df = pd.read_sql(query, engine)

pd.options.display.float_format = '{:.0f}'.format

df_avg = df.groupby('state', as_index=False)['st_population'].mean().sort_values(by='st_population', ascending=False)
df_avg.head(10)



,state,st_population
18,Madhya Pradesh,14316431
19,Maharashtra,8801923
24,Odisha,8256017
14,Jharkhand,7866055
10,Gujarat,7520036
27,Rajasthan,7270374
6,Chhattisgarh,7219749
1,Andhra Pradesh,5047219
33,West Bengal,4504169
3,Assam,3355794


In [28]:
df_avg = df.groupby('state', as_index=False)['st_share_state_population_pct'].mean().sort_values(by='st_share_state_population_pct', ascending=False)
df_avg.head(10)

,state,st_share_state_population_pct
17,Lakshadweep,95
22,Mizoram,94
23,Nagaland,86
21,Meghalaya,86
2,Arunachal Pradesh,69
7,Dadra and Nagar Haveli and Daman and Diu,58
20,Manipur,35
28,Sikkim,34
30,Tripura,32
6,Chhattisgarh,31


In [31]:
df_avg = df.groupby('state', as_index=False)['decadal_growth_pct'].mean().sort_values(by='decadal_growth_pct', ascending=False)
df_avg.head(10)

,state,decadal_growth_pct
18,Madhya Pradesh,15
19,Maharashtra,10
24,Odisha,9
27,Rajasthan,9
10,Gujarat,9
14,Jharkhand,8
6,Chhattisgarh,8
1,Andhra Pradesh,6
33,West Bengal,5
15,Karnataka,4


In [10]:
df_avg = df[['state', 'year', 'total_population', 'st_population']].copy()

df_avg = df_avg.dropna(subset=['total_population', 'st_population'])
df_avg = df_avg[df_avg['total_population'] > 0]

df_avg = df_avg.groupby('state', as_index=False).agg({
    'total_population': 'mean',
    'st_population': 'mean',
    'year': 'max'
})

df_avg['st_population_pct'] = (
    df_avg['st_population'] / df_avg['total_population']
) * 100

df_avg['st_population_pct'] = df_avg['st_population_pct'].round(2)

df_avg = df_avg.sort_values(
    by=['st_population_pct', 'st_population'],
    ascending=[False, False]
)

df_avg[['state', 'year', 'st_population_pct']].head(15)

plot_df = df_avg[['state', 'st_population_pct']].head(15).sort_values('st_population_pct')

plt.figure(figsize=(10, 7))
plt.barh(plot_df['state'], plot_df['st_population_pct'])
plt.xlabel('ST Population (%)')
plt.ylabel('State')
plt.title('Top 15 States by ST Population Percentage')
plt.tight_layout()
plt.show()

KeyError: "['year'] not in index"

In [8]:
def explore_table(table_name):
    df = pd.read_sql(f"SELECT * FROM {table_name}", engine)
    
    print(f"\n=== {table_name} ===")
    print(df.head())
    print("\nColumns:", df.columns.tolist())
    print("\nMissing values:\n", df.isnull().sum())
    print("\nShape:", df.shape)
    
    return df

In [41]:
df_literacy = explore_table("literacy")
df_poverty = explore_table("poverty_st")
df_employment = explore_table("employment_st")
df_demographics = explore_table("demographics_st")
df_dropout = explore_table("dropout_st")
df_low_literacy_districts = explore_table("low_literacy_districts")
df_household_type_rural = explore_table("household_type_rural")
df_high_st_share = explore_table("high_st_share")
df_ger_st = explore_table("ger_st")
df_enrolment_st = explore_table("enrolment_st")
df_mgnreg = explore_table("mgnreg_st")
df_sc_residence = explore_table("sc_st_residence")
df_tribal_villages = explore_table("tribal_villages")
df_tribe_socioeconomic = explore_table("tribe_socioeconomic")


=== literacy ===
                         state  year additional_info  total_literacy_rate_pct  \
0  Andaman and Nicobar Islands  2011            None                       87   
1               Andhra Pradesh  2011            None                       67   
2            Arunachal Pradesh  2011            None                       65   
3                        Assam  2011            None                       72   
4                        Bihar  2011            None                       62   

   st_literacy_rate_pct  literacy_gap_pct  
0                    76                11  
1                    49                18  
2                    65                 1  
3                    72                 0  
4                    51                11  

Columns: ['state', 'year', 'additional_info', 'total_literacy_rate_pct', 'st_literacy_rate_pct', 'literacy_gap_pct']

Missing values:
 state                       0
year                        0
additional_info            90
total

In [4]:
from pathlib import Path

csv_files = [
     r"C:\Users\hadap\Downloads\DSM_Project\Dataset\state_analysis_dataset_high_st_states.csv",
     r"C:\Users\hadap\Downloads\DSM_Project\Dataset\state_analysis_dataset_all_states.csv"
]

for file_path in csv_files:
    table_name = Path(file_path).stem
    df = pd.read_csv(file_path)
    df.to_sql(table_name, con=engine, if_exists="replace", index=False)



In [2]:
import os
import re
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sqlalchemy import text

load_dotenv()

DB_SCHEMA = "dsm_final_project"
DEFAULT_MODEL = "tencent/hy3-preview:free"


def get_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Set OPENAI_API_KEY in your .env file before running the agent.")

    return OpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1"
    )


def list_tables(schema_name=DB_SCHEMA):
    query = text(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = :schema_name
        ORDER BY table_name
        """
    )
    return pd.read_sql(query, engine, params={"schema_name": schema_name})


def describe_table(table_name, schema_name=DB_SCHEMA):
    query = text(
        """
        SELECT
            column_name,
            data_type,
            is_nullable,
            column_key
        FROM information_schema.columns
        WHERE table_schema = :schema_name
          AND table_name = :table_name
        ORDER BY ordinal_position
        """
    )
    return pd.read_sql(
        query,
        engine,
        params={"schema_name": schema_name, "table_name": table_name},
    )


def build_schema_context(schema_name=DB_SCHEMA):
    table_names = list_tables(schema_name)["table_name"].tolist()
    sections = []

    for table_name in table_names:
        columns_df = describe_table(table_name, schema_name)
        column_lines = [
            f"- {row.column_name} ({row.data_type}, nullable={row.is_nullable}, key={row.column_key or 'none'})"
            for row in columns_df.itertuples(index=False)
        ]
        sections.append(f"Table: {table_name}\n" + "\n".join(column_lines))

    return "\n\n".join(sections)


def clean_sql(sql_text):
    sql_text = sql_text.strip()
    sql_text = re.sub(r"^```sql\s*|^```\s*|\s*```$", "", sql_text, flags=re.IGNORECASE)
    return sql_text.strip()


def is_safe_select_query(query):
    cleaned = clean_sql(query)
    lowered = cleaned.lower()
    lowered = re.sub(r"--.*?$", "", lowered, flags=re.MULTILINE)
    lowered = re.sub(r"/\*.*?\*/", "", lowered, flags=re.DOTALL)
    stripped = lowered.strip().rstrip(";")

    if not stripped.startswith(("select", "with")):
        return False

    blocked_terms = [
        "insert ",
        "update ",
        "delete ",
        "drop ",
        "alter ",
        "truncate ",
        "create ",
        "replace ",
        "grant ",
        "revoke ",
    ]
    return not any(term in stripped for term in blocked_terms)


def safe_run_sql(query, row_limit=200):
    cleaned = clean_sql(query).rstrip(";")
    if not is_safe_select_query(cleaned):
        raise ValueError("Only read-only SELECT queries are allowed.")

    limited_query = f"SELECT * FROM ({cleaned}) AS analyst_query LIMIT {row_limit}"
    return pd.read_sql(text(limited_query), engine)


def generate_sql(question, schema_context, model=DEFAULT_MODEL):
    client = get_openai_client()
    prompt = f"""
You are a careful MySQL data analyst.

Database schema:
{schema_context}

User question:
{question}

Rules:
- Return only SQL.
- Use only tables and columns from the schema.
- Produce a single read-only query.
- Never use INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, or TRUNCATE.
- Prefer explicit column names instead of SELECT * unless it is necessary.
- If aggregation is needed, include clear aliases.
"""

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "system",
                "content": [{"type": "input_text", "text": "Return only valid MySQL SQL."}],
            },
            {
                "role": "user",
                "content": [{"type": "input_text", "text": prompt}],
            },
        ],
    )
    return clean_sql(response.output_text)


def summarize_results(question, sql_query, result_df, model=DEFAULT_MODEL):
    client = get_openai_client()
    preview_csv = result_df.head(20).to_csv(index=False)

    prompt = f"""
You are summarizing the output of a SQL analysis.

User question:
{question}

SQL used:
{sql_query}

Preview of result rows:
{preview_csv}

Write a concise answer grounded only in the result preview.
If the preview is insufficient, say that more rows may be needed.
"""

    response = client.responses.create(
        model=model,
        input=prompt,
    )
    return response.output_text.strip()


def ask_database(question, model=DEFAULT_MODEL, row_limit=200, return_summary=True):
    schema_context = build_schema_context()
    sql_query = generate_sql(question, schema_context, model=model)
    result_df = safe_run_sql(sql_query, row_limit=row_limit)

    response = {
        "question": question,
        "sql_query": sql_query,
        "rows_returned": len(result_df),
        "data": result_df,
    }

    if return_summary:
        response["summary"] = summarize_results(question, sql_query, result_df, model=model)

    return response


# Example:
# result = ask_database("Which states have the highest literacy_total_pct?")
# print(result["summary"])
# print(result["sql_query"])
# result["data"].head()


In [9]:
result = ask_database("Which states have the highest literacy_total_pct?")
print(result["summary"])
print(result["sql_query"])
result["data"].head()


The states appearing at the very top of the ranking are:

1. **Kerala** – 94.0 % (2011)  
2. **Lakshadweep** – 91.8 % (2011)  
3. **Mizoram** – 91.3 % (2011)  

These have the highest total literacy percentages in the previewed results.
SELECT
    state,
    year,
    total_literacy_rate_pct AS literacy_total_pct
FROM
    literacy
ORDER BY
    total_literacy_rate_pct DESC
LIMIT 10;


,state,year,literacy_total_pct
0,Kerala,2011,94.0
1,Lakshadweep,2011,91.8
2,Mizoram,2011,91.3
3,Kerala,2001,90.9
4,Kerala,1991,89.8


In [17]:
result = ask_database("Return the poverty headcount ratio (poverty_headcount_ratio) for all states in the most recent year available, sorted from highest to lowest.")
print(result["summary"])
print(result["sql_query"])
result["data"].head()

For the most recent year available (the maximum year in the `poverty_st` table), the preview shows 19 states' poverty headcount ratios (average `st_bpl_pct`) sorted from highest to lowest:
- Highest: Odisha (51.6)
- Next: West Bengal (47.3), Chhattisgarh (43.900000000000006), Madhya Pradesh (43.8), Maharashtra (42.45), Jharkhand (40.15), Bihar (34.8), Gujarat (33.3), Karnataka (32.25), Rajasthan (31.549999999999997), Kerala (27.3), Assam (24.5), Uttar Pradesh (21.65), Tamil Nadu (19.799999999999997), Uttarakhand (18.8), Andhra Pradesh (18.1), Telangana (18.1), Jammu and Kashmir (9.65), Ladakh (9.65)
- Lowest in preview: Himachal Pradesh (6.75)

More rows may exist in the full result beyond this preview.
SELECT
    state,
    AVG(st_bpl_pct) AS poverty_headcount_ratio
FROM
    poverty_st
WHERE
    year = (SELECT MAX(year) FROM poverty_st)
GROUP BY
    state
ORDER BY
    poverty_headcount_ratio DESC;


,state,poverty_headcount_ratio
0,Odisha,51.60
1,West Bengal,47.30
2,Chhattisgarh,43.90
3,Madhya Pradesh,43.80
4,Maharashtra,42.45


In [4]:
from pathlib import Path

engine = create_engine("mysql+mysqlconnector://root:@localhost:3306/dsm_final_project")

csv_files = [
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\scholarships_st.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\mgnreg_st_dependency.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\high_st_share.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\ger_st_latest.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\ger_st_gender_summary.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\state_analysis_dataset_high_st_states.csv",
    r"C:\Users\hadap\Downloads\DSM_Project\Dataset\state_analysis_dataset_all_states.csv"
]

for file_path in csv_files:
    table_name = Path(file_path).stem
    df = pd.read_csv(file_path)
    df.to_sql(table_name, con=engine, if_exists="replace", index=False)

engine.dispose()